# Practical 06 – Classification I

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = "Alexa_dataset.csv"
LOCAL_CANDIDATES = [Path("data") / DATA_FILE, Path("Lecture06_Classification_I") / "data" / DATA_FILE]
GITHUB_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture06_Classification_I/data/Alexa_dataset.csv"
local_path = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
data_source = local_path if local_path else GITHUB_URL
df = pd.read_csv(data_source)
print(f"Loaded from: {data_source}")
df.head()

## Source Chapter Exercise – Identify inputs and output

The author classifier chapter asks students to inspect these attributes:

- User ID
- Gender
- Age
- Minutes of Music Consumed
- Purchased Alexa

### Questions

1. Which attribute is the target?
2. Which attributes should be used as input features?
3. Which attributes should not be used, and why?
4. Is this classification or regression?

### Source answer

- **Target:** `Purchased Alexa`
- **Inputs used in the chapter:** `Age` and `Minutes of Music Consumed`
- **User ID:** removed because it is only an identifier
- **Gender:** could be encoded, but the source example intentionally leaves it out for simplicity
- **Problem type:** binary classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = df[["Age", "Minutes of music consumed"]]
y = df["Purchased Alexa"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression())]),
    "KNN": Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))])
}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print("\n", name)
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall   :", recall_score(y_test, pred))
    print("F1       :", f1_score(y_test, pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))

## Exercise
Change `n_neighbors` and explain how K affects KNN.

## Source Chapter Visualization – Decision Boundaries

The author chapter visualizes the training and test decision regions for each two-feature classifier. The helper below works directly with our Scikit-Learn pipelines, so the scaling step remains inside the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_decision_boundary(model, X_plot, y_plot, title):
    x1_min, x1_max = X_plot.iloc[:, 0].min() - 2, X_plot.iloc[:, 0].max() + 2
    x2_min, x2_max = X_plot.iloc[:, 1].min() - 5000, X_plot.iloc[:, 1].max() + 5000

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 250),
        np.linspace(x2_min, x2_max, 250)
    )

    grid = pd.DataFrame({
        X_plot.columns[0]: xx1.ravel(),
        X_plot.columns[1]: xx2.ravel()
    })

    zz = model.predict(grid).reshape(xx1.shape)

    plt.contourf(xx1, xx2, zz, alpha=0.25)
    plt.scatter(
        X_plot.iloc[:, 0],
        X_plot.iloc[:, 1],
        c=y_plot,
        edgecolor="k"
    )
    plt.xlabel(X_plot.columns[0])
    plt.ylabel(X_plot.columns[1])
    plt.title(title)
    plt.show()

for name, clf in models.items():
    clf.fit(X_train, y_train)
    plot_decision_boundary(clf, X_train, y_train, f"{name} – Training Set")
    plot_decision_boundary(clf, X_test, y_test, f"{name} – Test Set")